# Playbook Soccer Analytics — GPU Test (Google Colab)

**Before running:** Go to `Runtime → Change runtime type` and select **T4 GPU**.

You will need:
- A **Roboflow API key** (free at roboflow.com) stored in Colab Secrets as `ROBOFLOW_API_KEY`
- A short soccer video clip (MP4, ideally 10–30 seconds for a quick test)

**Steps:**
1. Run cells 1–4 to set up the environment (one-time, ~3–5 min)
2. Run cell 5 to add your API key
3. Run cell 6 to provide your video
4. Run cell 7 to process and see results

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU detected:', gpu.stdout.strip())
else:
    print('⚠️  No GPU found.\n'
          'Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 2: Install system packages ───────────────────────────────────────────
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [ ]:
# ── Cell 3: Install Python dependencies ───────────────────────────────────────
# Colab ships opencv-python; swap to headless to avoid display conflicts
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null

!pip install -q \
    numpy==1.26.4 \
    opencv-python-headless==4.8.0.76 \
    tqdm \
    requests \
    pydantic==2.8.2 \
    pydantic-settings==2.4.0 \
    python-dotenv==1.0.1 \
    supervision==0.22.0 \
    inference==0.16.2

# CUDA-enabled PyTorch (matches torch==2.4.1 in requirements.txt, CUDA 12.1)
!pip install -q \
    torch==2.4.1+cu121 \
    torchvision==0.19.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    transformers==4.55.4 \
    tokenizers==0.21.4 \
    huggingface_hub==0.34.0 \
    safetensors==0.4.5

# Roboflow sports library (SigLIP team classifier)
!pip install -q git+https://github.com/roboflow/sports.git@main

print('\n✅ All packages installed.')

In [ ]:
# ── Cell 4: Clone the repo ─────────────────────────────────────────────────────
# If the repo is private, use a GitHub personal access token:
#   !git clone https://<YOUR_TOKEN>@github.com/muwafagq/playbook-program.git /content/playbook
# If public:
!git clone https://github.com/muwafagq/playbook-program.git /content/playbook

import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 5: API key setup ─────────────────────────────────────────────────────
# Option A (recommended): Store key in Colab Secrets
#   Left sidebar → 🔑 Secrets → Add new secret → Name: ROBOFLOW_API_KEY
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print('✅ Loaded API key from Colab Secrets')
except Exception:
    # Option B: Paste directly (less secure)
    ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY_HERE'
    print('⚠️  Using hardcoded API key — prefer Colab Secrets for security')

os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
os.environ['DEVICE'] = 'cuda'   # use GPU for team classification
print('Device set to: cuda')

In [ ]:
# ── Cell 6: Provide a test video ──────────────────────────────────────────────
# Choose ONE option below and comment out the others.

# --- Option A: Upload a local file -------------------------------------------
from google.colab import files as colab_files
print('Select your MP4 video file in the dialog below...')
uploaded = colab_files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print('Video ready at:', VIDEO_PATH)

# --- Option B: Download from YouTube (yt-dlp) --------------------------------
# Paste a YouTube URL of any publicly available soccer clip (10–30 s recommended)
# !pip install -q yt-dlp
# YT_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
# !yt-dlp -o /content/test_clip.%(ext)s --recode-video mp4 -q "$YT_URL"
# VIDEO_PATH = '/content/test_clip.mp4'
# print('Video ready at:', VIDEO_PATH)

# --- Option C: Mount Google Drive and use a file already stored there --------
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_clip.mp4'
# print('Video ready at:', VIDEO_PATH)

In [ ]:
# ── Cell 7 (optional): Trim video to first N seconds ─────────────────────────
# Skip this cell if your clip is already short (< 30 s).
TRIM_SECONDS = 20   # adjust as needed
TRIMMED_PATH = '/content/test_trimmed.mp4'
!ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c copy "{TRIMMED_PATH}" -loglevel warning
VIDEO_PATH = TRIMMED_PATH
print(f'Trimmed to {TRIM_SECONDS}s → {VIDEO_PATH}')

In [ ]:
# ── Cell 8: Run the pipeline ──────────────────────────────────────────────────
# enable_team=True  → team classification ON  (uses GPU, ~2–4x slower)
# enable_team=False → team classification OFF (faster, still does detection + tracking + homography)
# Recommendation: start with False for a quick sanity check, then set True.

from main import main

main(
    source_video=VIDEO_PATH,
    out_dir='/content/outputs',
    enable_team=True,
    fit_team_stride=30,
    fit_team_max_frames=60,   # cap team-fitting to 60 frames for speed
)

In [ ]:
# ── Cell 9: Preview the annotated video ───────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode

video_bytes = open('/content/outputs/annotated.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(video_bytes).decode()
HTML(f'''
<video width="800" controls>
  <source src="{data_url}" type="video/mp4">
</video>
''')

In [ ]:
# ── Cell 10: Preview tracking CSV ─────────────────────────────────────────────
import pandas as pd
df = pd.read_csv('/content/outputs/per_frame_tracks.csv')
print(f'Rows: {len(df):,}  |  Frames: {df.frame.nunique()}  |  Unique tracks: {df.track_id.nunique()}')
df.head(10)

In [ ]:
# ── Cell 11: Download outputs ─────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download('/content/outputs/annotated.mp4')
colab_files.download('/content/outputs/per_frame_tracks.csv')